# Question 3: Graph-Based Social Network Analysis

## Introduction

Social networks map naturally onto **graphs**: users are vertices, friendships are (undirected) edges. This notebook represents a small social network as an **adjacency list**, traverses it, and computes mutual friends.

## (a) Adjacency List Representation

An adjacency list is a dictionary mapping each user to the list of users they are directly connected to. It is memory-efficient — O(V + E) space — because only *existing* edges are stored, unlike an adjacency matrix which stores O(V²) entries regardless of how sparse the graph is.

A second, disconnected pair (Eve–Frank) is included deliberately, to allow a genuine test of "no mutual friends" / "no path exists" cases later — most real social graphs are not fully connected.

In [1]:
graph = {
    "Victor": ["Alice", "Bob"],
    "Alice": ["Victor", "Bob"],
    "Bob": ["Victor", "Alice", "David"],
    "David": ["Bob"],
    "Eve": ["Frank"],     # a separate, disconnected component
    "Frank": ["Eve"]
}

for user, friends in graph.items():
    print(f"{user}: {len(friends)} friend(s) -> {friends}")

Victor: 2 friend(s) -> ['Alice', 'Bob']
Alice: 2 friend(s) -> ['Victor', 'Bob']
Bob: 3 friend(s) -> ['Victor', 'Alice', 'David']
David: 1 friend(s) -> ['Bob']
Eve: 1 friend(s) -> ['Frank']
Frank: 1 friend(s) -> ['Eve']


## (b) BFS Traversal

Breadth-First Search explores the graph level by level using a **queue**, making it ideal for finding shortest connection paths (e.g. "degrees of separation") in a social network.

In [2]:
from collections import deque

def bfs(graph, start):
    visited = {start}
    order = []
    queue = deque([start])
    while queue:
        node = queue.popleft()
        order.append(node)
        for neighbour in graph[node]:
            if neighbour not in visited:
                visited.add(neighbour)
                queue.append(neighbour)
    return order

bfs_order = bfs(graph, "Victor")
print("BFS traversal from Victor:", bfs_order)
assert bfs_order == ["Victor", "Alice", "Bob", "David"]
print("Matches expected order ✔  |  Time complexity: O(V + E)")

BFS traversal from Victor: ['Victor', 'Alice', 'Bob', 'David']
Matches expected order ✔  |  Time complexity: O(V + E)


DFS (Depth-First Search) is included for completeness since the question allows either BFS *or* DFS — both are implemented and tested here.

In [3]:
def dfs(graph, start):
    visited = set()
    order = []
    def _walk(node):
        visited.add(node)
        order.append(node)
        for neighbour in graph[node]:
            if neighbour not in visited:
                _walk(neighbour)
    _walk(start)
    return order

dfs_order = dfs(graph, "Victor")
print("DFS traversal from Victor:", dfs_order)
reachable_from_victor = {"Victor", "Alice", "Bob", "David"}
assert set(dfs_order) == reachable_from_victor, "DFS must visit every vertex reachable from the start node"
print("Visits all reachable vertices ✔  |  Time complexity: O(V + E)")

DFS traversal from Victor: ['Victor', 'Alice', 'Bob', 'David']
Visits all reachable vertices ✔  |  Time complexity: O(V + E)


## (c) Finding Mutual Friends Using Python Sets

Mutual friends are the **set intersection** of two users' friend lists. Python's `&` operator computes this in O(min(len(set1), len(set2))) on average — far better than a nested-loop comparison.

In [4]:
def mutual_friends(graph, user1, user2):
    return set(graph[user1]) & set(graph[user2])

mf = mutual_friends(graph, "Victor", "Alice")
print(f"Mutual friends of Victor and Alice: {mf}")
assert mf == {"Bob"}

# Edge case: users with no mutual friends (Eve belongs to a separate component)
mf2 = mutual_friends(graph, "Victor", "Eve")
print(f"Mutual friends of Victor and Eve: {mf2}")
assert mf2 == set()
print("\nVERIFIED ✔")

Mutual friends of Victor and Alice: {'Bob'}
Mutual friends of Victor and Eve: set()

VERIFIED ✔


## (d) Displaying the Full Graph Structure

Traversing the adjacency list and printing each user with their direct connections gives a complete, human-readable snapshot of the network.

In [5]:
def display_graph(graph):
    print("=" * 40)
    print("SOCIAL NETWORK GRAPH STRUCTURE")
    print("=" * 40)
    for user, friends in graph.items():
        print(f"{user:8} -> {friends}")
    print("=" * 40)
    total_edges = sum(len(f) for f in graph.values()) // 2
    print(f"Vertices: {len(graph)}, Edges: {total_edges}")

display_graph(graph)

SOCIAL NETWORK GRAPH STRUCTURE
Victor   -> ['Alice', 'Bob']
Alice    -> ['Victor', 'Bob']
Bob      -> ['Victor', 'Alice', 'David']
David    -> ['Bob']
Eve      -> ['Frank']
Frank    -> ['Eve']
Vertices: 6, Edges: 5


### Complexity Summary

| Operation | Complexity |
|---|---|
| BFS / DFS Traversal | O(V + E) |
| Mutual Friends (set intersection) | O(min(\|F₁\|, \|F₂\|)) average |
| Display Full Graph | O(V + E) |

Where V = number of users (vertices) and E = number of friendships (edges). This representation and these algorithms scale well to large, sparse social graphs, which is the typical structure of real-world social networks.